# 🎬 AI Video Studio — Free Colab GPU Worker

This notebook turns Google Colab's **free T4 GPU** into a generation backend for your local **AI Video Studio** (`app.py`).

**How it works:** this notebook loads **Wan2.1-T2V-1.3B** on the Colab GPU and starts a tiny Gradio server with a public `*.gradio.live` URL. You paste that URL into the studio's **⚙️ Settings & System** tab, and every video you generate locally is rendered on Colab's GPU — at full model quality, for free.

### Steps
1. **Runtime → Change runtime type → T4 GPU → Save**
2. Run every cell top to bottom (`Runtime → Run all`).
3. Copy the `https://xxxxx.gradio.live` URL printed by the **last** cell.
4. In the local studio: **⚙️ Settings & System → Google Colab Worker URL** → paste → **💾 Save API Keys & Settings**.
5. Keep this Colab tab open while you generate. Free sessions idle-out after a while — just re-run if the URL dies.

> First run downloads the model weights (~17 GB) to the Colab session; this takes a few minutes and is normal.

## 1. Verify the GPU

In [ ]:
import os
import subprocess

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
import torch

assert torch.cuda.is_available(), 'No GPU! Runtime > Change runtime type > T4 GPU'
print('GPU:', torch.cuda.get_device_name(0),
      '| VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')

## 2. Install dependencies + fetch the Wan2.1 code

In [ ]:
import importlib

%cd /content
![ -d Wan2.1 ] || git clone https://github.com/Wan-Video/Wan2.1.git
%cd /content/Wan2.1
!grep -viE '^flash[_-]?attn' requirements.txt > /tmp/wan_reqs.txt
!pip install -q -r /tmp/wan_reqs.txt
# gradio>=5 so the worker's API protocol matches the local gradio_client (2.5.0).
# The old 4.44.1 pin was several protocol revisions behind the client and risked
# a broken /generate handshake. After launching, verify the round-trip locally
# with:  python video_studio.py --probe  (Colab URL saved in Settings first).
!pip install -q "gradio>=5,<7" huggingface_hub ftfy dashscope imageio-ffmpeg
importlib.invalidate_caches()
for _pkg in ('easydict', 'ftfy', 'dashscope', 'imageio_ffmpeg', 'gradio'):
    importlib.import_module(_pkg)
print('deps installed | gradio:', importlib.import_module('gradio').__version__)

## 3. Download the T2V-1.3B weights (once per session)

In [ ]:
from huggingface_hub import snapshot_download

CKPT_DIR = '/content/Wan2.1-T2V-1.3B'
snapshot_download('Wan-AI/Wan2.1-T2V-1.3B', local_dir=CKPT_DIR)
print('weights ready at', CKPT_DIR)

## 4. Load the model once
Cached in memory so every request is fast.

In [ ]:
import gc
import os
import sys
from contextlib import nullcontext

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
sys.path.insert(0, '/content/Wan2.1')

import torch
import wan
import wan.modules.attention as _attn
import wan.modules.model as _model
from wan.configs import WAN_CONFIGS

if not (_attn.FLASH_ATTN_2_AVAILABLE or _attn.FLASH_ATTN_3_AVAILABLE):
    try:
        from torch.nn.attention import SDPBackend
        from torch.nn.attention import sdpa_kernel as _sdpa_kernel

        def _fast_sdp_ctx():
            return _sdpa_kernel([SDPBackend.FLASH_ATTENTION, SDPBackend.EFFICIENT_ATTENTION])
    except Exception:
        def _fast_sdp_ctx():
            return nullcontext()

    def _sdpa_fallback(q, k, v, **kwargs):
        out_dtype = q.dtype
        if 'version' in kwargs:
            kwargs['fa_version'] = kwargs.pop('version')
        ql = kwargs.get('q_lens')
        if ql is not None and ql.numel() == 1 and int(ql[0]) == q.size(1):
            kwargs['q_lens'] = None
        if kwargs.get('q_lens') is None:
            kl = kwargs.get('k_lens')
            if kl is not None and kl.numel() == 1 and int(kl[0]) == k.size(1):
                kwargs['k_lens'] = None
        target_dtype = kwargs.get('dtype') or out_dtype
        # T4 (SM 7.5) Cutlass EFFICIENT_ATTENTION only runs on fp16/fp32, not bf16;
        # without casting here PyTorch silently uses MATH (~57.5 GiB OOM on 89f).
        if target_dtype == torch.bfloat16 and q.is_cuda and torch.cuda.get_device_capability(q.device)[0] < 8:
            kwargs['dtype'] = torch.float16
        else:
            kwargs.setdefault('dtype', out_dtype)

        large_dense = (
            q.is_cuda
            and kwargs.get('q_lens') is None
            and kwargs.get('k_lens') is None
            and q.size(1) * k.size(1) > 4_000_000
        )
        if large_dense:
            try:
                with _fast_sdp_ctx():
                    return _attn.attention(q, k, v, **kwargs).type(out_dtype)
            except RuntimeError:
                if not kwargs.get('causal', False) and kwargs.get('window_size', (-1, -1)) == (-1, -1) and q.size(1) > 1024:
                    chunks = [
                        _attn.attention(q[:, i:i + 1024], k, v, **kwargs).type(out_dtype)
                        for i in range(0, q.size(1), 1024)
                    ]
                    return torch.cat(chunks, dim=1)
                raise
        return _attn.attention(q, k, v, **kwargs).type(out_dtype)

    _attn.flash_attention = _sdpa_fallback
    _model.flash_attention = _sdpa_fallback
    print('flash-attn unavailable; using PyTorch SDPA fallback')

CKPT_DIR = '/content/Wan2.1-T2V-1.3B'
cfg = WAN_CONFIGS['t2v-1.3B']
print('Loading Wan2.1-T2V-1.3B (offloaded to fit T4 16GB)...')
MODEL = wan.WanT2V(
    config=cfg, checkpoint_dir=CKPT_DIR, device_id=0,
    rank=0, t5_fsdp=False, dit_fsdp=False, use_usp=False,
    t5_cpu=True,          # keep the text encoder on CPU -> fits the 16GB T4
)
MODEL.vae.model.to('cpu')
_orig_vae_decode = MODEL.vae.decode


def _offloaded_vae_decode(zs):
    try:
        MODEL.vae.model.to(MODEL.device)
        return _orig_vae_decode(zs)
    finally:
        MODEL.vae.model.to('cpu')
        gc.collect()
        torch.cuda.empty_cache()


MODEL.vae.decode = _offloaded_vae_decode
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
with torch.no_grad():
    _q = torch.zeros((1, 35880, 12, 128), device='cuda', dtype=torch.bfloat16)
    _k = torch.zeros((1, 35880, 12, 128), device='cuda', dtype=torch.bfloat16)
    _v = torch.zeros((1, 35880, 12, 128), device='cuda', dtype=torch.bfloat16)
    _kl = torch.tensor([35880], device='cuda', dtype=torch.int32)
    _out = _attn.flash_attention(_q, _k, _v, k_lens=_kl)
    assert _out.shape == _q.shape and _out.dtype == torch.bfloat16
    del _q, _k, _v, _kl, _out
gc.collect()
torch.cuda.empty_cache()
print('SDPA 35880-token test passed | peak VRAM MB:',
      round(torch.cuda.max_memory_allocated() / 1e6))
torch.cuda.reset_peak_memory_stats()
print('model loaded')

## 5. Start the worker server
Exposes `/generate` with the exact signature the local `ColabBackend` calls:
`(prompt, negative_prompt, width, height, num_frames, steps, guidance, seed)`.

In [ ]:
import gc
import os
import tempfile

import gradio as gr
import torch
from wan.utils.utils import cache_video


def generate(prompt, negative_prompt, width, height, num_frames,
             steps, guidance, seed):
    width, height = int(width), int(height)
    num_frames = int(num_frames)
    if (num_frames - 1) % 4 != 0:            # Wan needs 4n+1 frames
        num_frames = (num_frames // 4) * 4 + 1
    seed = int(seed)
    if seed < 0:
        seed = torch.randint(0, 2**31 - 1, (1,)).item()
    print(f'gen {width}x{height} f{num_frames} s{steps} seed{seed}: {prompt[:60]}',
          flush=True)
    video = None
    out = tempfile.NamedTemporaryFile(suffix='.mp4', delete=False).name
    try:
        video = MODEL.generate(
            prompt,
            size=(width, height),
            frame_num=num_frames,
            sampling_steps=int(steps),
            guide_scale=float(guidance),
            n_prompt=negative_prompt or '',
            seed=seed,
            offload_model=True,                  # frees VRAM between stages on T4
        )
        saved = cache_video(video[None], save_file=out, fps=16,
                            nrow=1, normalize=True, value_range=(-1, 1))
        if not saved or not os.path.isfile(out) or os.path.getsize(out) <= 0:
            raise RuntimeError(f'cache_video failed to write non-empty MP4 to {out}')
    except Exception:
        if os.path.exists(out):
            os.remove(out)
        raise
    finally:
        del video
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    print(f'done {os.path.getsize(out)} bytes', flush=True)
    return out

with gr.Blocks(title='Wan2.1 Colab Worker') as worker:
    gr.Markdown('### Wan2.1 T2V-1.3B worker — leave this tab open')
    with gr.Row():
        p  = gr.Textbox(label='prompt')
        np_ = gr.Textbox(label='negative_prompt')
    with gr.Row():
        w = gr.Number(832, label='width');  h = gr.Number(480, label='height')
        nf = gr.Number(81, label='num_frames'); st = gr.Number(30, label='steps')
        g = gr.Number(6.0, label='guidance'); sd = gr.Number(-1, label='seed')
    vid = gr.Video(label='result')
    gr.Button('Generate', variant='primary').click(
        generate, [p, np_, w, h, nf, st, g, sd], vid, api_name='generate')

worker.queue(max_size=20)
print('\n' + '='*70)
print('COPY THE  https://xxxxx.gradio.live  URL BELOW into the local studio:')
print('='*70)
worker.launch(share=True, show_error=True)